# Advanced AI Lab 1 — Sentiment Analysis

This is the **single entry point** for all Lab 1 experiments.  
Run the cells top-to-bottom in each section you need.  
All heavy logic lives in the project files — this notebook just calls them.

| Section | Task |
|---|---|
| 0 — Setup | Check GPU, install dependencies, wandb login |
| 1 — Task 1.1 ANN | Simple ANN on small (1 K) and large (25 K) datasets |
| 2 — Task 1.1 BiLSTM | Bidirectional LSTM on small and large datasets |
| 3 — Task 1.2 BERT | Fine-tune BERT on the large Amazon dataset |
| 4 — Task 1.2 DistilBERT | Fine-tune DistilBERT on the large Amazon dataset |
| 5 — Grade 5 | BERT + DistilBERT on the public ~1 GB amazon_polarity dataset |
| 6 — Comparison | Side-by-side results across all models |
| 7 — W&B | View training curves at https://wandb.ai |

---
## Section 0 — Setup

Run this cell once before anything else.  
It verifies PyTorch, shows the GPU (if available), and confirms all modules load correctly.

In [1]:
import sys
import os

# Make sure the Lab 1 root is on the Python path so all imports resolve
project_root = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, project_root)

import torch
import config

print("=" * 55)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Device in use   : {config.DEVICE}")
print("=" * 55)
print()
print("Datasets found:")
print(f"  Small (1 K)  : {os.path.exists(config.SMALL_DATASET_PATH)}  →  {config.SMALL_DATASET_PATH}")
print(f"  Large (25 K) : {os.path.exists(config.LARGE_DATASET_PATH)}  →  {config.LARGE_DATASET_PATH}")
print()
print("All project modules are ready.")
print("Run any section below to start an experiment.")

PyTorch version : 2.11.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2080 Ti
Device in use   : cuda

Datasets found:
  Small (1 K)  : True  →  /Labs/Lab1/amazon_cells_labelled.txt
  Large (25 K) : True  →  /Labs/Lab1/amazon_cells_labelled_LARGE_25K.txt

All project modules are ready.
Run any section below to start an experiment.


### Section 0.1 — Weights & Biases Login (run once)

All experiments log to **https://wandb.ai** automatically.  
Run the cell below to install wandb (if missing) and authenticate.  
Get your API key at **https://wandb.ai/authorize**

In [2]:
import subprocess, sys

import wandb
wandb.login()   # prompts for API key if not already stored

print()
print("Logged in to wandb.")
print("After running experiments, view all results at: https://wandb.ai")
print(f"Project name: {__import__('config').WANDB_PROJECT}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: deeplab15group (deeplab15group-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Logged in to wandb.
After running experiments, view all results at: https://wandb.ai
Project name: advanced-ai-lab-1


---
## Section 1 — Task 1.1: Simple ANN

A feed-forward network trained on **TF-IDF bag-of-words features**.  
We run the same architecture twice — once on the 1 K small dataset, once on the 25 K large dataset — to observe how training-data volume impacts accuracy.

| Cell | Dataset | What we learn |
|---|---|---|
| 1.1 | Small (1 K) | Baseline — limited data |
| 1.2 | Large (25 K) | Effect of 25× more training data |

In [3]:
# 1.1  Simple ANN — small dataset (1 K reviews)
from experiments.task01_ann import main as run_ann

ann_small = run_ann(dataset="small")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [small dataset] …
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 4,033 features
  Split: 700 train / 150 val / 150 test

Building ANN  (vocab_size=4,033) …
  [build_ann] dataset='small' → SmallANN (2 blocks, 128→64)
  Total parameters    : 524,994
  Trainable parameters: 524,994



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Small
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.0005
  Epochs      : 50
  Patience    : 3
  Trainable   : 524,994 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/50  |  train  loss=0.7163  acc=50.9%  |  val  loss=0.6931  acc=50.0%  f1=0.6606  prec=0.5000  rec=0.9733


  Epoch  2/50  |  train  loss=0.6865  acc=56.1%  |  val  loss=0.6741  acc=56.7%  f1=0.6632  prec=0.5424  rec=0.8533


  Epoch  3/50  |  train  loss=0.6364  acc=64.6%  |  val  loss=0.6316  acc=72.0%  f1=0.7407  prec=0.6897  rec=0.8000


...

  Epoch  8/50  |  train  loss=0.0740  acc=98.6%  |  val  loss=0.4922  acc=78.0%  f1=0.7925  prec=0.7500  rec=0.8400


  Epoch  9/50  |  train  loss=0.0662  acc=98.9%  |  val  loss=0.4925  acc=77.3%  f1=0.7901  prec=0.7356  rec=0.8533
  Early stopping triggered (no improvement for 3 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 73.33%   F1 : 0.7436   Precision : 0.7160   Recall : 0.7733
──────────────────────────────────────────────────────────────



batch_loss,▇██▇███▇▇▇▇▆▆▆▇▇▆▅▅▆▅▅▅▄▃▃▃▂▂▄▂▂▂▁▂▁▁▁▁▁
epoch,▁▂▃▄▅▅▆▇█
train_accuracy,▁▂▃▅▆▇███
train_loss,██▇▆▅▃▂▁▁
val_accuracy,▁▃▇▇█████
val_f1,▁▁▅▅▇█▇██
val_loss,█▇▆▄▂▁▁▂▂
val_precision,▁▂▆▇█████
val_recall,█▄▂▁▂▃▂▃▄
batch_loss,0.04079
epoch,9


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Small.pth
[Result] Experiment       : Task01_ANN_Small
[Result] Best Val Accuracy: 78.00%
[Result] Test Accuracy    : 73.33%
[Result] Test F1          : 0.7436



In [4]:
# 1.2  Simple ANN — large dataset (25 K reviews)
from experiments.task01_ann import main as run_ann

ann_large = run_ann(dataset="large")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [large dataset] …
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 11,619 features
  Split: 17,510 train / 3,740 val / 3,750 test

Building ANN  (vocab_size=11,619) …
  [build_ann] dataset='large' → LargeANN (3 blocks, 512→256→64)
  Total parameters    : 6,098,882
  Trainable parameters: 6,098,882



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Large
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=5e-05
  Epochs      : 20
  Patience    : 2
  Trainable   : 6,098,882 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/20  |  train  loss=0.6539  acc=60.3%  |  val  loss=0.5606  acc=75.5%  f1=0.8268  prec=0.7222  rec=0.9668


  Epoch  2/20  |  train  loss=0.4810  acc=78.1%  |  val  loss=0.3708  acc=85.8%  f1=0.8804  prec=0.8953  rec=0.8660


  Epoch  3/20  |  train  loss=0.3047  acc=88.6%  |  val  loss=0.3173  acc=86.6%  f1=0.8870  prec=0.9033  rec=0.8713


  Epoch  4/20  |  train  loss=0.1993  acc=92.8%  |  val  loss=0.3245  acc=86.2%  f1=0.8834  prec=0.9030  rec=0.8647


  Epoch  5/20  |  train  loss=0.1231  acc=96.1%  |  val  loss=0.3555  acc=85.6%  f1=0.8797  prec=0.8896  rec=0.8700
  Early stopping triggered (no improvement for 2 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 86.37%   F1 : 0.8858   Precision : 0.8980   Recall : 0.8738
──────────────────────────────────────────────────────────────



batch_loss,██▇▇▇▇█▇▇▆▆▆▆▅▅▅▄▄▄▃▂▂▃▂▂▂▁▂▂▂▂▁▁▁▁▂▁▂▂▁
epoch,▁▃▅▆█
train_accuracy,▁▄▇▇█
train_loss,█▆▃▂▁
val_accuracy,▁▇██▇
val_f1,▁▇██▇
val_loss,█▃▁▁▂
val_precision,▁███▇
val_recall,█▁▁▁▁
batch_loss,0.14431
epoch,5


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Large.pth
[Result] Experiment       : Task01_ANN_Large
[Result] Best Val Accuracy: 86.58%
[Result] Test Accuracy    : 86.37%
[Result] Test F1          : 0.8858



In [3]:
# grade 5  Simple ANN — public dataset (~100 K reviews from amazon_polarity)
from experiments.task01_ann import main as run_ann

ann_public = run_ann(dataset="public")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading ANN data  [public dataset] …
  (This may take a while on first run — the dataset is ~1 GB)


  Loaded 3,600,000 examples from 'amazon_polarity'
  Preprocessing text (classical) …
  Fitting TF-IDF on training split (will NOT see val / test) …
  TF-IDF vocab: 30,000 features
  Split: 2,521,440 train / 538,560 val / 540,000 test

Building ANN  (vocab_size=30,000) …
  [build_ann] dataset='public' → LargeANN (3 blocks, 512→256→64)
  Total parameters    : 15,509,954
  Trainable parameters: 15,509,954



══════════════════════════════════════════════════════════════
  Experiment  : Task01_ANN_Public
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.0001
  Epochs      : 10
  Patience    : 2
  Trainable   : 15,509,954 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/10  |  train  loss=0.2871  acc=87.5%  |  val  loss=0.2286  acc=90.8%  f1=0.9080  prec=0.9059  rec=0.9101


  Epoch  2/10  |  train  loss=0.2448  acc=90.3%  |  val  loss=0.2208  acc=91.2%  f1=0.9119  prec=0.9090  rec=0.9148


...

  Epoch  9/10  |  train  loss=0.2048  acc=92.2%  |  val  loss=0.1959  acc=92.3%  f1=0.9227  prec=0.9268  rec=0.9187


  Epoch 10/10  |  train  loss=0.1885  acc=93.0%  |  val  loss=0.1948  acc=92.4%  f1=0.9230  prec=0.9319  rec=0.9144



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 92.46%   F1 : 0.9239   Precision : 0.9329   Recall : 0.9151
──────────────────────────────────────────────────────────────



batch_loss,█▃▃▂▂▂▂▃▂▂▃▂▂▂▂▂▂▃▃▃▂▂▃▃▂▂▂▁▃▁▂▂▁▃▂▃▃▃▂▂
epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▅▅▅▅▆▆▆▇█
train_loss,█▅▅▄▄▄▃▃▂▁
val_accuracy,▁▃▃▄▅▅▆▇██
val_f1,▁▃▃▄▅▆▆▇██
val_loss,█▆▆▅▅▄▃▂▁▁
val_precision,▁▂▄▃▃▄▆▆▇█
val_recall,▁▄▁▄█▅▅▆▆▄
batch_loss,0.11497
epoch,10


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_ANN_Public.pth
[Result] Experiment       : Task01_ANN_Public
[Result] Best Val Accuracy: 92.38%
[Result] Test Accuracy    : 92.46%
[Result] Test F1          : 0.9239



---
## Section 2 — Task 1.1: Bidirectional LSTM

A BiLSTM with **learned word embeddings** that reads the sentence in both directions.  
Unlike the ANN, the BiLSTM preserves word order and captures long-range dependencies  
(e.g. negations, emphasis) — important signals for sentiment.

| Cell | Dataset | What we learn |
|---|---|---|
| 2.1 | Small (1 K) | Sequence model on limited data |
| 2.2 | Large (25 K) | Sequence model vs ANN on the same 25 K data |

In [3]:
# 2.1  BiLSTM — small dataset (1 K reviews)
from experiments.task01_bilstm import main as run_bilstm

bilstm_small = run_bilstm(dataset="small")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading BiLSTM data  [small dataset] …
  Preprocessing text (classical) …
  Building vocabulary from training split only …
  Vocabulary size: 1,418 tokens (max 30,000)
  Split: 686 train / 147 val / 147 test

Building BiLSTMSentiment  (vocab_size=1,418) …
  Total parameters    : 95,810
  Trainable parameters: 95,810



══════════════════════════════════════════════════════════════
  Experiment  : Task01_BiLSTM_Small
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.003
  Epochs      : 50
  Patience    : 7
  Trainable   : 95,810 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/50  |  train  loss=0.6940  acc=51.5%  |  val  loss=0.6904  acc=52.4%  f1=0.6569  prec=0.5154  rec=0.9054


  Epoch  2/50  |  train  loss=0.6933  acc=52.9%  |  val  loss=0.6840  acc=61.2%  f1=0.7016  prec=0.5726  rec=0.9054


  Epoch  3/50  |  train  loss=0.6811  acc=57.9%  |  val  loss=0.6701  acc=60.5%  f1=0.6947  prec=0.5690  rec=0.8919


...

  Epoch 36/50  |  train  loss=0.1018  acc=95.8%  |  val  loss=0.7002  acc=77.6%  f1=0.7815  prec=0.7662  rec=0.7973


  Epoch 37/50  |  train  loss=0.1297  acc=93.7%  |  val  loss=0.6844  acc=78.9%  f1=0.7891  prec=0.7945  rec=0.7838
  Early stopping triggered (no improvement for 7 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 67.35%   F1 : 0.6712   Precision : 0.6712   Recall : 0.6712
──────────────────────────────────────────────────────────────



batch_loss,▅▅▅▅▅▄▅▄▃▃▃▃▄▄▃▃▂▂▃▁▂█▃▂▃▂▁▂▁▂▁▁▃▂▁▁▁▂▁▁
epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train_accuracy,▁▁▂▃▃▃▄▄▅▅▅▆▅▆▆▇▆▇▇▇▇▇▇▇▇▇▇▄▇▇███████
train_loss,▇▇▆▆▆▆▅▅▅▄▄▄▅▄▃▃▃▃▃▂▃▂▂▂▂▂▂█▃▂▂▁▁▁▂▁▁
val_accuracy,▁▃▃▃▅▄▆▅▅▆▆▆▆▆▇▆▆▆▆▇▇▇▇▇▇▇▆▇▇█▇██▇███
val_f1,▁▃▃▃▂▃▄▅▅▅▅▆▆▅▇▆▅▆▆▆▇█▇▇▇▇▆▇▇█▇█▇▇▇▇█
val_loss,▅▅▄▄▄▃▂▃▃▂▃▃▂▂▃▄▃▄▄▃▄▄▃▅▄▄█▄▁▂▄▃▅▄▅▅▅
val_precision,▁▂▂▂▄▃▆▅▄▆▄▅▅▆▅▅▅▅▅▆▆▆▇▆▇▆▄▅▇█▆█▇▇▇▇█
val_recall,▇▇▇▇▁▅▂▅▆▃▆▆▆▃▆▆▄▅▅▄▆▆▅▆▅▆█▆▄▄▆▄▄▄▄▄▃
batch_loss,0.18918
epoch,37


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_BiLSTM_Small.pth
[Result] Experiment       : Task01_BiLSTM_Small
[Result] Best Val Accuracy: 78.91%
[Result] Test Accuracy    : 67.35%
[Result] Test F1          : 0.6712



TF-IDF beats learned embeddings on tiny data — document this in your report and move on to the large dataset where BiLSTM should outperform the ANN.

In [6]:
# 2.2  BiLSTM — large dataset (25 K reviews)
from experiments.task01_bilstm import main as run_bilstm

bilstm_large = run_bilstm(dataset="large")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading BiLSTM data  [large dataset] …
  Preprocessing text (classical) …
  Building vocabulary from training split only …
  Vocabulary size: 21,312 tokens (max 30,000)
  Split: 17,510 train / 3,740 val / 3,750 test

Building BiLSTMSentiment  (vocab_size=21,312) …
  Total parameters    : 5,096,450
  Trainable parameters: 5,096,450



══════════════════════════════════════════════════════════════
  Experiment  : Task01_BiLSTM_Large
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.001
  Epochs      : 50
  Patience    : 5
  Trainable   : 5,096,450 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/50  |  train  loss=0.6656  acc=59.7%  |  val  loss=0.5902  acc=68.9%  f1=0.7451  prec=0.7372  rec=0.7532


  Epoch  2/50  |  train  loss=0.5875  acc=68.1%  |  val  loss=0.5035  acc=75.3%  f1=0.7771  prec=0.8571  rec=0.7107


  Epoch  3/50  |  train  loss=0.5318  acc=73.0%  |  val  loss=0.4461  acc=78.1%  f1=0.8068  prec=0.8645  rec=0.7563


...

  Epoch 20/50  |  train  loss=0.2098  acc=92.2%  |  val  loss=0.3497  acc=85.2%  f1=0.8830  prec=0.8437  rec=0.9261


  Epoch 21/50  |  train  loss=0.1935  acc=92.9%  |  val  loss=0.3528  acc=86.0%  f1=0.8858  prec=0.8729  rec=0.8992
  Early stopping triggered (no improvement for 5 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 85.20%   F1 : 0.8803   Precision : 0.8615   Recall : 0.8999
──────────────────────────────────────────────────────────────



batch_loss,██▇█▅▆▇▆▆▅▅▅▄▅▆▄▆▅▅▅▃▄▆▅▃▄▄▃▃▃▁▄▄▂▂▂▁▁▃▁
epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
train_accuracy,▁▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇████
train_loss,█▇▆▆▅▅▅▄▄▄▃▃▃▃▂▂▃▂▁▁▁
val_accuracy,▁▄▅▅▆▆▆▇▇▇█████████▇█
val_f1,▁▃▄▄▆▅▆▇▇▇███████████
val_loss,█▆▄▄▃▄▃▂▂▂▁▁▁▁▁▁▁▁▁▂▂
val_precision,▁▆▆▇▅▅▇▅▇▇▇▇▇█▇▇▇▇▆▆▇
val_recall,▂▁▂▃▆▅▅▇▆▆▇▇▇▆▇▇▇▇██▇
batch_loss,0.09172
epoch,21


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_BiLSTM_Large.pth
[Result] Experiment       : Task01_BiLSTM_Large
[Result] Best Val Accuracy: 86.55%
[Result] Test Accuracy    : 85.20%
[Result] Test F1          : 0.8803



The BiLSTM large is within 0.2% of the ANN large — effectively identical. This is actually a meaningful finding:
On 25K samples, BiLSTM and ANN are comparable. The BiLSTM's advantage from learned embeddings is offset by its higher parameter count needing more data to train well. The real separation will come at the public dataset scale (100K samples) where embeddings shine.

In [7]:
# grade 5  BiLSTM — public dataset (3.6M  reviews from amazon_polarity)
from experiments.task01_bilstm import main as run_bilstm

bilstm_public = run_bilstm(dataset="public")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading BiLSTM data  [public dataset] …
  (This may take a while on first run — the dataset is ~1 GB)
  Loaded 3,600,000 examples from 'amazon_polarity'
  Preprocessing text (classical) …
  Building vocabulary from training split only …
  Vocabulary size: 30,000 tokens (max 30,000)
  Split: 2,521,440 train / 538,560 val / 540,000 test

Building BiLSTMSentiment  (vocab_size=30,000) …
  Total parameters    : 6,208,514
  Trainable parameters: 6,208,514



══════════════════════════════════════════════════════════════
  Experiment  : Task01_BiLSTM_Public
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : Adam  lr=0.001
  Epochs      : 10
  Patience    : 3
  Trainable   : 6,208,514 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/10  |  train  loss=0.3507  acc=84.0%  |  val  loss=0.2378  acc=90.4%  f1=0.9033  prec=0.9052  rec=0.9015


  Epoch  2/10  |  train  loss=0.2429  acc=90.1%  |  val  loss=0.2960  acc=87.3%  f1=0.8844  prec=0.8122  rec=0.9707


  Epoch  3/10  |  train  loss=0.2346  acc=90.5%  |  val  loss=0.2224  acc=91.1%  f1=0.9108  prec=0.9099  rec=0.9118


...

  Epoch  9/10  |  train  loss=0.2224  acc=91.1%  |  val  loss=0.2670  acc=89.8%  f1=0.9041  prec=0.8517  rec=0.9634


  Epoch 10/10  |  train  loss=0.2158  acc=91.4%  |  val  loss=0.2053  acc=91.8%  f1=0.9184  prec=0.9144  rec=0.9224



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 91.84%   F1 : 0.9187   Precision : 0.9153   Recall : 0.9222
──────────────────────────────────────────────────────────────



batch_loss,█▇▇▂▃▅▃▄▂▄▂▁▃▃▂▆▃▆▃▃▃▁▄▅▂▂▄▂▂▂▃▂▂▃▃▂▃▄▃▃
epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▇▇▇▇█████
train_loss,█▂▂▂▂▂▁▁▁▁
val_accuracy,▆▁▇▇▇▇▇▇▅█
val_f1,▅▁▆▆▇▇▇▇▅█
val_loss,▄█▂▂▂▂▂▂▆▁
val_precision,▇▁▇█▆▇▆█▃▇
val_recall,▁█▂▁▄▄▅▂▇▃
batch_loss,0.22136
epoch,10


Checkpoint saved → /Labs/Lab1/checkpoints/Task01_BiLSTM_Public.pth
[Result] Experiment       : Task01_BiLSTM_Public
[Result] Best Val Accuracy: 91.80%
[Result] Test Accuracy    : 91.84%
[Result] Test F1          : 0.9187



---
## Section 3 — Task 1.2: BERT Fine-Tuning

**BERT-base-uncased** was pre-trained on BookCorpus + English Wikipedia using  
masked language modelling and next-sentence prediction.  
We replace its classification head and fine-tune **all 12 transformer layers** for sentiment.

> ⚠ GPU strongly recommended — BERT has 110 M parameters.  
> On CPU, reduce `"epochs"` in `config.py → BERT_CONFIG` to `1`.

In [3]:
# 3.1  BERT — small dataset (1 K reviews)
from experiments.task02_bert import main as run_bert

bert_small = run_bert(dataset="small")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading transformer data  [small dataset, bert-base-uncased] …
  Preprocessing text (minimal, transformer mode) …
  Tokenising with 'bert-base-uncased' (max_len=128) …


  Split: 686 train / 147 val / 147 test

Building bert-base-uncased (pretrained on BookCorpus + Wikipedia) …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total parameters    : 109,483,778
  Trainable parameters: 109,483,778


...

  Epoch  7/15  |  train  loss=0.0192  acc=99.9%  |  val  loss=0.1631  acc=95.2%  f1=0.9530  prec=0.9467  rec=0.9595


  Epoch  8/15  |  train  loss=0.0172  acc=99.9%  |  val  loss=0.1672  acc=95.2%  f1=0.9530  prec=0.9467  rec=0.9595
  Early stopping triggered (no improvement for 5 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 91.84%   F1 : 0.9155   Precision : 0.9420   Recall : 0.8904
──────────────────────────────────────────────────────────────



batch_loss,█▇▆▇▆▇▆▅▄▄▃▃▄▂▂▅▂▁▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▂▃▄▅▆▇█
train_accuracy,▁▅▇▇████
train_loss,█▆▃▃▂▁▁▁
val_accuracy,▁▇██████
val_f1,▁▆█▇████
val_loss,█▃▁▁▁▁▁▁
val_precision,▁▇███▇▇▇
val_recall,▁▂▇▅▇███
batch_loss,0.01008
epoch,8


Checkpoint saved → /Labs/Lab1/checkpoints/Task02_BERT_Small.pth
[Result] Experiment       : Task02_BERT_Small
[Result] Best Val Accuracy: 95.92%
[Result] Test Accuracy    : 91.84%
[Result] Test F1          : 0.9155



In [4]:
# 3.2  BERT — large dataset (25 K reviews)
from experiments.task02_bert import main as run_bert

bert_large = run_bert(dataset="large")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading transformer data  [large dataset, bert-base-uncased] …
  Preprocessing text (minimal, transformer mode) …
  Tokenising with 'bert-base-uncased' (max_len=128) …
  Split: 17,510 train / 3,740 val / 3,750 test

Building bert-base-uncased (pretrained on BookCorpus + Wikipedia) …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total parameters    : 109,483,778
  Trainable parameters: 109,483,778



══════════════════════════════════════════════════════════════
  Experiment  : Task02_BERT_Large
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : AdamW  lr=2e-05
  Epochs      : 5
  Patience    : 3
  Trainable   : 109,483,778 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



...

  Epoch  3/5  |  train  loss=0.0722  acc=97.9%  |  val  loss=0.2137  acc=93.7%  f1=0.9481  prec=0.9510  rec=0.9452


  Epoch  4/5  |  train  loss=0.0420  acc=98.9%  |  val  loss=0.2652  acc=93.4%  f1=0.9455  prec=0.9397  rec=0.9513
  Early stopping triggered (no improvement for 3 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 92.75%   F1 : 0.9402   Precision : 0.9365   Recall : 0.9440
──────────────────────────────────────────────────────────────



batch_loss,███▄▂▃▅▃▇▃▃▂▄▄▂▄▂▂▁▂▂▂▁▁▁▁▁▁▃▁▁▁▁▄▄▂▃▁▁▁
epoch,▁▃▆█
train_accuracy,▁▆▇█
train_loss,█▄▂▁
val_accuracy,█▂█▁
val_f1,█▂█▁
val_loss,▁▂▄█
val_precision,▇▁█▃
val_recall,▂█▁▄
batch_loss,0.00585
epoch,4


Checkpoint saved → /Labs/Lab1/checkpoints/Task02_BERT_Large.pth
[Result] Experiment       : Task02_BERT_Large
[Result] Best Val Accuracy: 93.74%
[Result] Test Accuracy    : 92.75%
[Result] Test F1          : 0.9402



In [ ]:
# 3.3  BERT — public dataset (~100 K reviews from amazon_polarity)
from experiments.task02_bert import main as run_bert

bert_public = run_bert(dataset="public")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading transformer data  [public dataset, bert-base-uncased] …
  (This may take a while on first run — the dataset is ~1 GB)


  Loaded 3,600,000 examples from 'amazon_polarity'
  Preprocessing text (minimal, transformer mode) …
  Loading tokeniser 'bert-base-uncased' (max_len=128) …
  Split: 2,521,440 train / 538,560 val / 540,000 test
  Tokenisation deferred to DataLoader (on-the-fly, no RAM pre-allocation)

Building bert-base-uncased (pretrained on BookCorpus + Wikipedia) …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total parameters    : 109,483,778
  Trainable parameters: 109,483,778



══════════════════════════════════════════════════════════════
  Experiment  : Task02_BERT_Public
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : AdamW  lr=3e-05
  Epochs      : 3
  Patience    : 2
  Trainable   : 109,483,778 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/3  |  train  loss=0.1177  acc=95.7%  |  val  loss=0.0974  acc=96.6%  f1=0.9658  prec=0.9678  rec=0.9638


  Epoch   2 [train]:   2%|▏         | 1283/78795 [05:38<5:39:52,  3.80it/s, acc=97.1%, loss=0.0839]

---
## Section 4 — Task 1.2: DistilBERT Fine-Tuning

**DistilBERT-base-uncased** is a knowledge-distilled version of BERT:  
40% fewer parameters, 60% faster, ~97% of BERT's accuracy.  

Running both side-by-side provides a direct **complexity vs accuracy trade-off** study.

In [4]:
# 4.1  DistilBERT — small dataset (1 K reviews)
from experiments.task02_distilbert import main as run_distilbert

distilbert_small = run_distilbert(dataset="small")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading transformer data  [small dataset, distilbert-base-uncased] …
  Preprocessing text (minimal, transformer mode) …
  Loading tokeniser 'distilbert-base-uncased' (max_len=128) …


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  Epoch 11/15  |  train  loss=0.0057  acc=99.9%  |  val  loss=0.1575  acc=95.9%  f1=0.9583  prec=0.9857  rec=0.9324


  Epoch 12/15  |  train  loss=0.0044  acc=99.9%  |  val  loss=0.1574  acc=95.9%  f1=0.9583  prec=0.9857  rec=0.9324
  Early stopping triggered (no improvement for 5 epochs).



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 94.56%   F1 : 0.9452   Precision : 0.9452   Recall : 0.9452
──────────────────────────────────────────────────────────────



batch_loss,███▆▃▃▂▂▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▂▂▃▄▄▅▅▆▇▇█
train_accuracy,▁▇▇█████████
train_loss,█▅▂▂▁▁▁▁▁▁▁▁
val_accuracy,▁▇▃▆▇▇██████
val_f1,▁▇▃▆▇▇██████
val_loss,█▁▃▁▁▂▁▁▁▁▁▁
val_precision,▂▂█▁██▇▇▇▇▇▇
val_recall,▁█▁▇▅▅▇▇▇▇▇▇
batch_loss,0.00266
epoch,12


Checkpoint saved → /Labs/Lab1/checkpoints/Task02_DistilBERT_Small.pth
[Result] Experiment       : Task02_DistilBERT_Small
[Result] Best Val Accuracy: 95.92%
[Result] Test Accuracy    : 94.56%
[Result] Test F1          : 0.9452



In [5]:
# 4.2  DistilBERT — large dataset (25 K reviews)
from experiments.task02_distilbert import main as run_distilbert

distilbert_large = run_distilbert(dataset="large")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading transformer data  [large dataset, distilbert-base-uncased] …
  Preprocessing text (minimal, transformer mode) …
  Loading tokeniser 'distilbert-base-uncased' (max_len=128) …
  Split: 17,510 train / 3,740 val / 3,750 test
  Tokenisation deferred to DataLoader (on-the-fly, no RAM pre-allocation)

Building distilbert-base-uncased (knowledge-distilled from BERT) …


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total parameters    : 66,955,010
  Trainable parameters: 66,955,010



══════════════════════════════════════════════════════════════
  Experiment  : Task02_DistilBERT_Large
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : AdamW  lr=2e-05
  Epochs      : 5
  Patience    : 3
  Trainable   : 66,955,010 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



...

  Epoch  4/5  |  train  loss=0.0625  acc=98.4%  |  val  loss=0.2420  acc=92.9%  f1=0.9414  prec=0.9444  rec=0.9385


  Epoch  5/5  |  train  loss=0.0438  acc=99.0%  |  val  loss=0.2557  acc=93.0%  f1=0.9422  prec=0.9405  rec=0.9438



──────────────────────────────────────────────────────────────
  [FINAL]  Test Accuracy : 92.56%   F1 : 0.9388   Precision : 0.9337   Recall : 0.9440
──────────────────────────────────────────────────────────────



batch_loss,█▃▂▄▃▄▂▂▃▃▂▂▂▅▃▂▄▁▂▃▁▂▃▁▂▁▂▂▁▃▃▁▁▃▁▃▂▂▁▁
epoch,▁▃▅▆█
train_accuracy,▁▆▇██
train_loss,█▄▂▁▁
val_accuracy,▁▅▇▇█
val_f1,▁▃█▇█
val_loss,▁▂▂▆█
val_precision,▅█▁▅▄
val_recall,▂▁█▅▅
batch_loss,0.00447
epoch,5


Checkpoint saved → /Labs/Lab1/checkpoints/Task02_DistilBERT_Large.pth
[Result] Experiment       : Task02_DistilBERT_Large
[Result] Best Val Accuracy: 92.99%
[Result] Test Accuracy    : 92.56%
[Result] Test F1          : 0.9388



In [ ]:
# 4.3  DistilBERT — public dataset (~100 K reviews from amazon_polarity)
from experiments.task02_distilbert import main as run_distilbert

distilbert_public = run_distilbert(dataset="public")


Device: cuda
GPU   : NVIDIA GeForce RTX 2080 Ti

Loading transformer data  [public dataset, distilbert-base-uncased] …
  (This may take a while on first run — the dataset is ~1 GB)
  Loaded 3,600,000 examples from 'amazon_polarity'
  Preprocessing text (minimal, transformer mode) …
  Loading tokeniser 'distilbert-base-uncased' (max_len=128) …
  Split: 2,521,440 train / 538,560 val / 540,000 test
  Tokenisation deferred to DataLoader (on-the-fly, no RAM pre-allocation)

Building distilbert-base-uncased (knowledge-distilled from BERT) …


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total parameters    : 66,955,010
  Trainable parameters: 66,955,010



══════════════════════════════════════════════════════════════
  Experiment  : Task02_DistilBERT_Public
  Device      : cuda  (NVIDIA GeForce RTX 2080 Ti)
  Optimiser   : AdamW  lr=3e-05
  Epochs      : 3
  Patience    : 2
  Trainable   : 66,955,010 parameters
  Split       : 70% train / 15% val / 15% test
  W&B project : advanced-ai-lab-1  (view at https://wandb.ai)
══════════════════════════════════════════════════════════════



  Epoch  1/3  |  train  loss=0.1227  acc=95.4%  |  val  loss=0.0990  acc=96.5%  f1=0.9652  prec=0.9658  rec=0.9647


  Epoch  2/3  |  train  loss=0.0827  acc=97.2%  |  val  loss=0.0939  acc=96.7%  f1=0.9671  prec=0.9705  rec=0.9637


  Epoch   3 [train]:   1%|          | 396/39398 [01:50<3:02:15,  3.57it/s, acc=98.0%, loss=0.0600]